In [5]:
pip install autogluon.tabular

  Using cached scikit_learn-1.7.2-cp313-cp313-macosx_12_0_arm64.whl.metadata (11 kB)
Using cached scikit_learn-1.7.2-cp313-cp313-macosx_12_0_arm64.whl (8.6 MB)
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.8.0
    Uninstalling scikit-learn-1.8.0:
      Successfully uninstalled scikit-learn-1.8.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
imbalanced-learn 0.13.0 requires sklearn-compat<1,>=0.1, which is not installed.
Note: you may need to restart the kernel to use updated packages.


In [1]:
"""
NCAA March Madness — Full Pipeline  (v9 — AutoGluon + HPO)
══════════════════════════════════════════════════════════════════════════
 WHAT THIS DOES
 AutoGluon automatically:
   • Trains 10+ model types: XGBoost, LightGBM, CatBoost, RandomForest,
     ExtraTrees, NeuralNet (PyTorch), KNN, LinearModel, FastAI NN
   • Runs Hyperparameter Optimisation (HPO) on every model type
   • Stacks models across multiple layers (L1 → L2 → L3 ensemble)
   • Picks optimal ensemble weights via weighted ensemble
   • Calibrates probabilities automatically
   • Reports leaderboard sorted by Brier score

 FEATURES (same proven 29 from v8 winning solution):
   seed, ELO, GLM quality, avg stats (score, FGA, OR, DR, Blk, PF,
   opponent FGA/Blk/PF, point differential)

 THREE PRESETS  (choose based on your time budget):
   PRESET = 'best_quality'    → ~20–60 min,  strongest model (recommended)
   PRESET = 'high_quality'    → ~5–15 min,   strong model
   PRESET = 'good_quality'    → ~2–5  min,   fast, decent accuracy

 DATA SPLIT (zero leakage)
 ├─ Train  Regular season 2013–2023 + Tournament 2013–2023
 └─ Test   Tournament 2024

 INSTALL
   pip install autogluon.tabular
══════════════════════════════════════════════════════════════════════════
"""

import pandas as pd
import numpy as np
import re
import os
import warnings
warnings.filterwarnings('ignore')
from itertools import combinations
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (brier_score_loss, log_loss,
                              roc_auc_score, accuracy_score,
                              confusion_matrix)

# ── Install AutoGluon if not present ─────────────────────────────────
try:
    from autogluon.tabular import TabularPredictor, TabularDataset
    print("AutoGluon already installed ✓")
except ImportError:
    print("Installing AutoGluon (this may take a few minutes)...")
    import subprocess, sys
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        'autogluon.tabular', '--quiet'
    ])
    from autogluon.tabular import TabularPredictor, TabularDataset
    print("AutoGluon installed ✓")

# ══════════════════════════════════════════════════════════════════════
# CONFIGURATION  ← change these to suit your machine
# ══════════════════════════════════════════════════════════════════════
BASE          = '/Users/shaurya/Downloads/'
TRAIN_SEASONS = list(range(2013, 2024))   # 2013–2023 inclusive
PRED_SEASON   = 2024
AG_SAVE_PATH  = '/Users/shaurya/Downloads/autogluon_ncaa_v9/'

# ── Preset options ─────────────────────────────────────────────────────
# 'best_quality'  → longest, strongest  (recommended for competition)
# 'high_quality'  → medium runtime, very good results
# 'good_quality'  → fast, still solid
PRESET        = 'best_quality'

# Time limit in seconds (AutoGluon stops training when hit)
# best_quality:  3600 (1 hr)  /  high_quality: 900 / good_quality: 300
TIME_LIMIT    = 3600

# HPO: number of trials per model type
# Higher = better tuning but slower. 10-20 is a good balance.
HPO_TRIALS    = 15

# ══════════════════════════════════════════════════════════════════════
# ░░  PHASE 1 — LOAD DATA  ░░
# ══════════════════════════════════════════════════════════════════════
print("=" * 70)
print("  NCAA March Madness v9 — AutoGluon + HPO")
print("=" * 70)
print(f"\n  Preset     : {PRESET}")
print(f"  Time limit : {TIME_LIMIT}s ({TIME_LIMIT//60} min)")
print(f"  HPO trials : {HPO_TRIALS} per model type")

print("\n[1/7] Loading data...")
m_reg     = pd.read_csv(BASE + 'MRegularSeasonDetailedResults.csv')
m_seeds   = pd.read_csv(BASE + 'MNCAATourneySeeds.csv')
m_tourney = pd.read_csv(BASE + 'MNCAATourneyDetailedResults.csv')

try:
    m_teams       = pd.read_csv(BASE + 'MTeams.csv')[['TeamID','TeamName']]
    team_name_map = m_teams.set_index('TeamID')['TeamName'].to_dict()
except Exception:
    team_name_map = {}

def tname(tid):
    return team_name_map.get(int(tid), str(int(tid)))

def parse_seed(s):
    m = re.search(r'\d+', str(s))
    return int(m.group()) if m else 16

m_seeds['SeedNum'] = m_seeds['Seed'].apply(parse_seed)

print(f"     Regular season rows : {len(m_reg):,}")
print(f"     Tournament rows     : {len(m_tourney):,}")
print(f"     Train seasons       : {TRAIN_SEASONS[0]}–{TRAIN_SEASONS[-1]}")
print(f"     Predict / Test      : {PRED_SEASON}")

# ══════════════════════════════════════════════════════════════════════
# ░░  PHASE 2 — ELO RATINGS  ░░
# ══════════════════════════════════════════════════════════════════════
print("\n[2/7] Computing ELO ratings (regular-season only, pre-tournament)...")

def compute_elo(reg_df, k=20, base=1500):
    elo, snap = {}, {}
    for season, sdf in (reg_df.sort_values(['Season','DayNum'])
                               .groupby('Season')):
        for t in list(elo):
            elo[t] = 0.75 * elo[t] + 0.25 * base
        for _, r in sdf.iterrows():
            w, l = r['WTeamID'], r['LTeamID']
            elo.setdefault(w, base)
            elo.setdefault(l, base)
            ew  = 1 / (1 + 10 ** ((elo[l] - elo[w]) / 400))
            mov = (np.log(max(abs(r['WScore'] - r['LScore']), 1) + 1)
                   * (2.2 / (ew * 0.001 + 2.2)))
            elo[w] += k * mov * (1 - ew)
            elo[l] += k * mov * (0 - (1 - ew))
        for t, v in elo.items():
            snap[(season, t)] = v
    return snap

elo_dict = compute_elo(m_reg[m_reg['Season'] <= PRED_SEASON])
print(f"     {len(elo_dict):,} (season, team) ELO values")

# ══════════════════════════════════════════════════════════════════════
# ░░  PHASE 3 — TEAM QUALITY (GLM / raddar method)  ░░
# ══════════════════════════════════════════════════════════════════════
print("[3/7] Computing GLM team quality scores (raddar method)...")

def compute_quality_scores(reg_df, seasons):
    quality = {}
    for season in seasons:
        sdf = reg_df[reg_df['Season'] == season]
        if sdf.empty:
            continue
        all_teams = sorted(
            pd.unique(sdf[['WTeamID','LTeamID']].values.ravel())
        )
        tid2idx = {t: i for i, t in enumerate(all_teams)}
        n       = len(all_teams)

        rows_X, rows_y = [], []
        for _, r in sdf.iterrows():
            w, l = r['WTeamID'], r['LTeamID']
            xw = np.zeros(n); xw[tid2idx[l]] = 1.0
            rows_X.append(xw); rows_y.append(1)
            xl = np.zeros(n); xl[tid2idx[w]] = 1.0
            rows_X.append(xl); rows_y.append(0)

        lr = LogisticRegression(
            C=1.0, fit_intercept=True,
            max_iter=500, random_state=42, solver='lbfgs'
        )
        lr.fit(np.array(rows_X), np.array(rows_y))
        coefs = lr.coef_[0]
        for i, tid in enumerate(all_teams):
            quality[(season, tid)] = float(coefs[i])
    return quality

all_seasons  = sorted(m_reg[m_reg['Season'] <= PRED_SEASON]['Season'].unique())
quality_dict = compute_quality_scores(m_reg, all_seasons)
print(f"     {len(quality_dict):,} (season, team) quality scores")

# ══════════════════════════════════════════════════════════════════════
# ░░  PHASE 4 — TEAM STATS  ░░
# ══════════════════════════════════════════════════════════════════════
print("[4/7] Building per-team season averages (regular-season only)...")

def build_team_season_stats(reg_df):
    w_cols = {
        'WTeamID':'TeamID','WScore':'Score','LScore':'OppScore',
        'WFGA':'FGA','LFGA':'opp_FGA',
        'WOR':'OR','WDR':'DR',
        'WBlk':'Blk','LBlk':'opp_Blk',
        'WPF':'PF','LPF':'opp_PF',
    }
    l_cols = {
        'LTeamID':'TeamID','LScore':'Score','WScore':'OppScore',
        'LFGA':'FGA','WFGA':'opp_FGA',
        'LOR':'OR','LDR':'DR',
        'LBlk':'Blk','WBlk':'opp_Blk',
        'LPF':'PF','WPF':'opp_PF',
    }
    common = ['Season','DayNum']
    w = reg_df[common + list(w_cols.keys())].rename(columns=w_cols).copy()
    l = reg_df[common + list(l_cols.keys())].rename(columns=l_cols).copy()
    long = pd.concat([w, l], ignore_index=True)
    long['PointDiff'] = long['Score'] - long['OppScore']
    stats = {}
    for (season, tid), grp in long.groupby(['Season','TeamID']):
        stats[(season, tid)] = {
            'avg_Score'        : grp['Score'].mean(),
            'avg_FGA'          : grp['FGA'].mean(),
            'avg_OR'           : grp['OR'].mean(),
            'avg_DR'           : grp['DR'].mean(),
            'avg_Blk'          : grp['Blk'].mean(),
            'avg_PF'           : grp['PF'].mean(),
            'avg_opponent_FGA' : grp['opp_FGA'].mean(),
            'avg_opponent_Blk' : grp['opp_Blk'].mean(),
            'avg_opponent_PF'  : grp['opp_PF'].mean(),
            'avg_PointDiff'    : grp['PointDiff'].mean(),
        }
    return stats

team_stats = build_team_season_stats(
    m_reg[m_reg['Season'] <= PRED_SEASON]
)

# Season medians for cold-start fallback
STAT_KEYS = ['avg_Score','avg_FGA','avg_OR','avg_DR','avg_Blk','avg_PF',
             'avg_opponent_FGA','avg_opponent_Blk','avg_opponent_PF','avg_PointDiff']
season_medians = {}
for season in all_seasons:
    vals = {k: [] for k in STAT_KEYS}
    for (s, _), v in team_stats.items():
        if s == season:
            for k in STAT_KEYS: vals[k].append(v[k])
    season_medians[season] = {k: np.median(v) if v else 0.0
                               for k, v in vals.items()}

def get_stats(season, team_id):
    return team_stats.get(
        (season, team_id),
        season_medians.get(season, {k: 0.0 for k in STAT_KEYS})
    )

print(f"     {len(team_stats):,} (season, team) stat profiles")

# ══════════════════════════════════════════════════════════════════════
# ░░  PHASE 5 — BUILD MATCHUP DATAFRAME  ░░
# ══════════════════════════════════════════════════════════════════════
print("[5/7] Building matchup rows (29-feature set)...")

# Exact 29 features from winning Kaggle solution
FEATURES = [
    "men_women",
    "T1_seed", "T2_seed", "Seed_diff",
    "T1_avg_Score", "T1_avg_FGA", "T1_avg_OR", "T1_avg_DR",
    "T1_avg_Blk", "T1_avg_PF",
    "T1_avg_opponent_FGA", "T1_avg_opponent_Blk", "T1_avg_opponent_PF",
    "T1_avg_PointDiff",
    "T2_avg_Score", "T2_avg_FGA", "T2_avg_OR", "T2_avg_DR",
    "T2_avg_Blk", "T2_avg_PF",
    "T2_avg_opponent_FGA", "T2_avg_opponent_Blk", "T2_avg_opponent_PF",
    "T2_avg_PointDiff",
    "T1_elo", "T2_elo", "elo_diff",
    "T1_quality", "T2_quality",
]
LABEL = "Label"

def make_row(season, t1, t2, seeds_df, label=None):
    s1 = seeds_df[(seeds_df['Season']==season)&(seeds_df['TeamID']==t1)]
    s2 = seeds_df[(seeds_df['Season']==season)&(seeds_df['TeamID']==t2)]
    seed1 = int(s1['SeedNum'].values[0]) if len(s1) else 16
    seed2 = int(s2['SeedNum'].values[0]) if len(s2) else 16
    ts1   = get_stats(season, t1)
    ts2   = get_stats(season, t2)
    e1    = elo_dict.get((season, t1), 1500.0)
    e2    = elo_dict.get((season, t2), 1500.0)
    q1    = quality_dict.get((season, t1), 0.0)
    q2    = quality_dict.get((season, t2), 0.0)
    row   = {
        "men_women"           : 1,
        "T1_seed"             : seed1,
        "T2_seed"             : seed2,
        "Seed_diff"           : seed1 - seed2,
        "T1_avg_Score"        : ts1['avg_Score'],
        "T1_avg_FGA"          : ts1['avg_FGA'],
        "T1_avg_OR"           : ts1['avg_OR'],
        "T1_avg_DR"           : ts1['avg_DR'],
        "T1_avg_Blk"          : ts1['avg_Blk'],
        "T1_avg_PF"           : ts1['avg_PF'],
        "T1_avg_opponent_FGA" : ts1['avg_opponent_FGA'],
        "T1_avg_opponent_Blk" : ts1['avg_opponent_Blk'],
        "T1_avg_opponent_PF"  : ts1['avg_opponent_PF'],
        "T1_avg_PointDiff"    : ts1['avg_PointDiff'],
        "T2_avg_Score"        : ts2['avg_Score'],
        "T2_avg_FGA"          : ts2['avg_FGA'],
        "T2_avg_OR"           : ts2['avg_OR'],
        "T2_avg_DR"           : ts2['avg_DR'],
        "T2_avg_Blk"          : ts2['avg_Blk'],
        "T2_avg_PF"           : ts2['avg_PF'],
        "T2_avg_opponent_FGA" : ts2['avg_opponent_FGA'],
        "T2_avg_opponent_Blk" : ts2['avg_opponent_Blk'],
        "T2_avg_opponent_PF"  : ts2['avg_opponent_PF'],
        "T2_avg_PointDiff"    : ts2['avg_PointDiff'],
        "T1_elo"              : e1,
        "T2_elo"              : e2,
        "elo_diff"            : e1 - e2,
        "T1_quality"          : q1,
        "T2_quality"          : q2,
    }
    if label is not None:
        row[LABEL] = label
    return row

def build_matchup_df(games_df, seeds_df):
    rows = []
    for _, g in games_df.iterrows():
        season = g['Season']
        t1, t2 = sorted([int(g['WTeamID']), int(g['LTeamID'])])
        label  = 1 if int(g['WTeamID']) == t1 else 0
        rows.append(make_row(season, t1, t2, seeds_df, label=label))
    return pd.DataFrame(rows)

# LEAKAGE GUARD: strictly Season ∈ TRAIN_SEASONS
train_tourney = m_tourney[m_tourney['Season'].isin(TRAIN_SEASONS)].copy()
reg_sample    = (m_reg[m_reg['Season'].isin(TRAIN_SEASONS)]
                 .sample(frac=0.30, random_state=42))

print("     Building tournament matchup rows...")
train_df = build_matchup_df(train_tourney, m_seeds)
print("     Building regular-season sample rows...")
reg_df   = build_matchup_df(reg_sample, m_seeds)
combined = pd.concat([train_df, reg_df], ignore_index=True)

# Hold out 2023 tournament as internal validation for AutoGluon
val_df   = combined[combined['T1_seed'].notna() &
                    train_df.index.isin(
                        train_df[train_tourney['Season'].values == 2023].index
                    )] if False else None
# Use a simple 10% holdout for AutoGluon's internal validation
from sklearn.model_selection import train_test_split
ag_train, ag_val = train_test_split(
    combined, test_size=0.10, random_state=42,
    stratify=combined[LABEL]
)

print(f"     Tournament matchups (2013-2023)  : {len(train_df):,}")
print(f"     Reg-season sample   (2013-2023)  : {len(reg_df):,}")
print(f"     AutoGluon train set              : {len(ag_train):,}")
print(f"     AutoGluon validation set         : {len(ag_val):,}")
print(f"     Features                         : {len(FEATURES)}")

# ══════════════════════════════════════════════════════════════════════
# ░░  PHASE 6 — AUTOGLUON TRAINING  ░░
# ══════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print(f"  PHASE 6 — AUTOGLUON  (preset={PRESET})")
print("=" * 70)

# ── HPO search spaces per model ───────────────────────────────────────
# AutoGluon will search these ranges automatically
from autogluon.core.space import Real, Int, Categorical

hpo_hyperparameters = {

    # XGBoost — full search around the known winning params
    'XGB': {
        'n_estimators'      : Int(200, 1200),
        'learning_rate'     : Real(0.005, 0.05, log=True),
        'max_depth'         : Int(3, 6),
        'min_child_weight'  : Int(2, 8),
        'subsample'         : Real(0.5, 0.9),
        'colsample_bytree'  : Real(0.5, 0.9),
        'reg_alpha'         : Real(0.0, 2.0),
        'reg_lambda'        : Real(0.5, 3.0),
        'ag_args': {'name_suffix': 'HPO'},
    },

    # LightGBM — wide search
    'GBM': {
        'num_leaves'        : Int(15, 63),
        'learning_rate'     : Real(0.005, 0.05, log=True),
        'n_estimators'      : Int(200, 1200),
        'min_child_samples' : Int(10, 40),
        'subsample'         : Real(0.5, 0.9),
        'colsample_bytree'  : Real(0.5, 0.9),
        'reg_alpha'         : Real(0.0, 1.5),
        'reg_lambda'        : Real(0.0, 2.0),
        'ag_args': {'name_suffix': 'HPO'},
    },

    # CatBoost
    'CAT': {
        'iterations'        : Int(200, 1000),
        'learning_rate'     : Real(0.005, 0.1, log=True),
        'depth'             : Int(4, 8),
        'l2_leaf_reg'       : Real(1.0, 10.0),
        'ag_args': {'name_suffix': 'HPO'},
    },

    # Random Forest
    'RF': {
        'n_estimators'      : Int(100, 500),
        'max_depth'         : Int(5, 20),
        'min_samples_leaf'  : Int(2, 10),
        'ag_args': {'name_suffix': 'HPO'},
    },

    # Extra Trees
    'XT': {
        'n_estimators'      : Int(100, 500),
        'max_depth'         : Int(5, 20),
        'min_samples_leaf'  : Int(2, 10),
        'ag_args': {'name_suffix': 'HPO'},
    },

    # Neural Net (PyTorch)
    'NN_TORCH': {
        'num_epochs'        : Int(20, 100),
        'learning_rate'     : Real(1e-4, 1e-2, log=True),
        'dropout_prob'      : Real(0.0, 0.5),
        'weight_decay'      : Real(1e-6, 1e-3, log=True),
        'ag_args': {'name_suffix': 'HPO'},
    },

    # Linear model (fast baseline, well-calibrated)
    'LR': {},
    'KNN': {'weights': Categorical('uniform','distance'),
            'n_neighbors': Int(3, 15)},
}

# ── HPO kwargs ────────────────────────────────────────────────────────
hpo_kwargs = {
    'searcher'   : 'auto',         # Bayesian search
    'scheduler'  : 'local',
    'num_trials' : HPO_TRIALS,     # per model type
}

print(f"\n  Starting AutoGluon fit...")
print(f"  Models to train  : XGB, LGB, CatBoost, RF, ExtraTrees,")
print(f"                     NeuralNet, KNN, LinearModel")
print(f"  Stacking levels  : 2 (base → stacker → weighted ensemble)")
print(f"  Bagging folds    : 8")
print(f"  HPO trials       : {HPO_TRIALS} per model type")
print(f"  Eval metric      : log_loss (maps to Brier minimisation)\n")

predictor = TabularPredictor(
    label       = LABEL,
    problem_type= 'binary',
    eval_metric = 'log_loss',        # best proxy for Brier in AutoGluon
    path        = AG_SAVE_PATH,
    verbosity   = 2,
)

predictor.fit(
    train_data   = TabularDataset(ag_train[FEATURES + [LABEL]]),
    tuning_data  = TabularDataset(ag_val[FEATURES + [LABEL]]),
    presets      = PRESET,
    time_limit   = TIME_LIMIT,
    hyperparameters          = hpo_hyperparameters,
    hyperparameter_tune_kwargs = hpo_kwargs,
    num_bag_folds            = 8,    # 8-fold bagging per model
    num_bag_sets             = 1,
    num_stack_levels         = 2,    # 2-level stacking
    refit_full               = True, # refit on full data after HPO
    feature_prune_kwargs     = None, # keep all 29 features
    ag_args_fit              = {'num_cpus': -1},  # use all cores
)

# ── AutoGluon leaderboard ─────────────────────────────────────────────
print("\n" + "─" * 70)
print("  AUTOGLUON LEADERBOARD")
print("─" * 70)
leaderboard = predictor.leaderboard(
    TabularDataset(ag_val[FEATURES + [LABEL]]),
    silent=True
)
print(leaderboard[['model','score_val','pred_time_val',
                   'fit_time','stack_level']].to_string(index=False))

best_model = leaderboard.iloc[0]['model']
best_score = leaderboard.iloc[0]['score_val']
print(f"\n  Best model    : {best_model}")
print(f"  Best val score: {best_score:.4f}  (log_loss — lower is better)")

# ── Feature importance from AutoGluon ─────────────────────────────────
print("\n  FEATURE IMPORTANCES (from best model)")
try:
    fi = predictor.feature_importance(
        data=TabularDataset(ag_val[FEATURES + [LABEL]]),
        silent=True
    )
    print(fi.head(29).to_string())
except Exception as e:
    print(f"  (Feature importance unavailable: {e})")

# ══════════════════════════════════════════════════════════════════════
# ░░  PHASE 7 — PREDICT 2024 TOURNAMENT  ░░
# ══════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print(f"  PHASE 7 — PREDICTING {PRED_SEASON} TOURNAMENT")
print("=" * 70)

seeds_pred = m_seeds[m_seeds['Season'] == PRED_SEASON]['TeamID'].unique()
pairs      = list(combinations(sorted(seeds_pred), 2))
print(f"     Teams : {len(seeds_pred)}  |  Pairs : {len(pairs):,}")

pred_rows = [make_row(PRED_SEASON, t1, t2, m_seeds) for t1, t2 in pairs]
pred_df   = pd.DataFrame(pred_rows)

# AutoGluon predict_proba returns DataFrame with columns [0, 1]
proba_df  = predictor.predict_proba(
    TabularDataset(pred_df[FEATURES])
)
# Column 1 = P(team1 wins)
raw_probs = proba_df[1].values
probs     = np.clip(raw_probs, 0.025, 0.975)

submission = pd.DataFrame({
    'ID'  : [f"{PRED_SEASON}_{t1}_{t2}" for t1, t2 in pairs],
    'Pred': probs,
})
sub_path = f'{BASE}submission_{PRED_SEASON}_v9_autogluon.csv'
submission.to_csv(sub_path, index=False)
print(f"     Submission → {sub_path}  ({len(submission):,} rows)")
print(f"     Pred range : {probs.min():.4f}–{probs.max():.4f}  "
      f"mean {probs.mean():.4f}")

# ── Print full prediction table ───────────────────────────────────────
seed_map = (m_seeds[m_seeds['Season'] == PRED_SEASON]
            .set_index('TeamID')['SeedNum'].to_dict())

print("\n" + "─" * 84)
print(f"  ALL {len(submission):,} MATCHUP PREDICTIONS  ({PRED_SEASON})")
print("─" * 84)
print(f"  {'ID':<28} {'Team 1':<28} {'Team 2':<28} {'Pred':>6}")
print("  " + "─" * 82)
for (t1, t2), p in zip(pairs, probs):
    s1 = seed_map.get(t1,'?'); s2 = seed_map.get(t2,'?')
    n1 = f"({s1}) {tname(t1)}"
    n2 = f"({s2}) {tname(t2)}"
    print(f"  {PRED_SEASON}_{t1}_{t2:<18}  {n1:<28} {n2:<28} {p:>6.4f}")
print("─" * 84)
print(f"  Total: {len(submission):,} pairs  |  Actual games played: 67")

# ══════════════════════════════════════════════════════════════════════
# ░░  VALIDATION AGAINST ACTUAL 2024 RESULTS  ░░
# ══════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print(f"  VALIDATION vs ACTUAL {PRED_SEASON} RESULTS")
print("=" * 70)

actual = m_tourney[m_tourney['Season'] == PRED_SEASON].copy()

if len(actual) == 0:
    print(f"  ⚠  No {PRED_SEASON} results found.")
else:
    def day_to_round(d):
        if d in range(134,136): return 'First Four'
        if d in range(136,138): return 'Round of 64'
        if d in range(138,140): return 'Round of 32'
        if d in range(143,146): return 'Sweet 16'
        if d in range(145,147): return 'Elite 8'
        if d in range(152,153): return 'Final Four'
        if d in range(154,155): return 'Championship'
        return 'Unknown'

    actual['Round'] = actual['DayNum'].apply(day_to_round)
    gt_rows = []
    for _, g in actual.iterrows():
        t1, t2  = sorted([int(g['WTeamID']), int(g['LTeamID'])])
        outcome = 1 if int(g['WTeamID']) == t1 else 0
        gt_rows.append({
            'ID'    : f"{PRED_SEASON}_{t1}_{t2}",
            'Actual': outcome,
            'Round' : g['Round'],
        })
    ground_truth = pd.DataFrame(gt_rows)
    val = ground_truth.merge(submission, on='ID', how='inner')
    print(f"\n  Matched games : {len(val)}  "
          f"(unmatched : {len(ground_truth)-len(val)})")

    if len(val) > 0:
        yt    = val['Actual'].values
        yp    = val['Pred'].values
        yb    = (yp >= 0.5).astype(int)
        brier = brier_score_loss(yt, yp)
        ll    = log_loss(yt, yp)
        auc   = roc_auc_score(yt, yp)
        acc   = accuracy_score(yt, yb)
        bl_b  = brier_score_loss(yt, np.full_like(yp, 0.5))
        skill = 1 - brier / bl_b

        print("\n" + "─" * 58)
        print("  OVERALL METRICS")
        print("─" * 58)
        print(f"  Games evaluated        : {len(val)}")
        print(f"  Brier Score            : {brier:.4f}  ← competition metric")
        print(f"  Baseline Brier (0.5)   : {bl_b:.4f}")
        print(f"  Brier Skill Score      : {skill:.4f}  (>0 beats naive)")
        print(f"  Log Loss               : {ll:.4f}")
        print(f"  ROC-AUC                : {auc:.4f}")
        print(f"  Accuracy (thresh=0.50) : {acc:.4f}  "
              f"({int(acc*len(val))}/{len(val)} correct)")
        print("─" * 58)

        cm = confusion_matrix(yt, yb)
        tn, fp, fn, tp = cm.ravel()
        prec = tp/(tp+fp) if tp+fp>0 else 0.0
        rec  = tp/(tp+fn) if tp+fn>0 else 0.0
        f1   = 2*prec*rec/(prec+rec) if prec+rec>0 else 0.0
        print(f"\n  Confusion Matrix (threshold 0.50)")
        print(f"               Pred 0   Pred 1")
        print(f"  Actual 0 :   {tn:>5}    {fp:>5}")
        print(f"  Actual 1 :   {fn:>5}    {tp:>5}")
        print(f"  Precision: {prec:.4f}   Recall: {rec:.4f}   F1: {f1:.4f}")

        # Calibration
        val['Bucket'] = pd.cut(
            val['Pred'],
            bins  =[0,.20,.35,.50,.65,.80,1.0],
            labels=['0–20%','20–35%','35–50%','50–65%','65–80%','80–100%']
        )
        calib = (val.groupby('Bucket', observed=True)
                    .agg(N=('Actual','count'),
                         AvgPred=('Pred','mean'),
                         ActWin=('Actual','mean'))
                    .reset_index())
        print(f"\n  CALIBRATION TABLE")
        print(f"  {'Bucket':<10} {'N':>5} {'AvgPred':>9} {'ActWin%':>9} {'Gap':>7}")
        print("  " + "─" * 44)
        for _, r in calib.iterrows():
            gap = r['AvgPred'] - r['ActWin']
            print(f"  {str(r['Bucket']):<10} {int(r['N']):>5} "
                  f"{r['AvgPred']:>9.3f} {r['ActWin']:>9.3f} {gap:>+7.3f}")

        # Round-by-round
        ROUNDS = ['First Four','Round of 64','Round of 32',
                  'Sweet 16','Elite 8','Final Four','Championship']
        print(f"\n  ROUND-BY-ROUND BREAKDOWN")
        print(f"  {'Round':<16} {'G':>4} {'Corr':>5} "
              f"{'Acc':>7} {'Brier':>7} {'AvgConf':>8}")
        print("  " + "─" * 56)
        for rnd in ROUNDS:
            sv = val[val['Round'] == rnd]
            if len(sv) == 0: continue
            yt_ = sv['Actual'].values
            yp_ = sv['Pred'].values
            yb_ = (yp_ >= 0.5).astype(int)
            print(f"  {rnd:<16} {len(sv):>4} "
                  f"{int(accuracy_score(yt_,yb_)*len(sv)):>5} "
                  f"{accuracy_score(yt_,yb_):>7.4f} "
                  f"{brier_score_loss(yt_,yp_):>7.4f} "
                  f"{yp_.mean():>8.3f}")

        # Seed-gap breakdown
        val['T1'] = val['ID'].apply(lambda x: int(x.split('_')[1]))
        val['T2'] = val['ID'].apply(lambda x: int(x.split('_')[2]))
        sm = (m_seeds[m_seeds['Season']==PRED_SEASON]
              .set_index('TeamID')['SeedNum'].to_dict())
        val['S1']        = val['T1'].apply(lambda x: sm.get(x,16))
        val['S2']        = val['T2'].apply(lambda x: sm.get(x,16))
        val['SeedDiff']  = abs(val['S1'] - val['S2'])
        val['SeedBucket']= pd.cut(
            val['SeedDiff'],
            bins  =[-1,2,5,9,20],
            labels=['Close (0–2)','Medium (3–5)','Large (6–9)','Blowout (10+)']
        )
        print(f"\n  BREAKDOWN BY SEED GAP")
        print(f"  {'Matchup':<18} {'G':>4} {'Corr':>5} "
              f"{'Acc':>7} {'Brier':>7}")
        print("  " + "─" * 44)
        for bucket, grp in val.groupby('SeedBucket', observed=True):
            if len(grp)==0: continue
            yt_ = grp['Actual'].values; yp_ = grp['Pred'].values
            print(f"  {str(bucket):<18} {len(grp):>4} "
                  f"{int(accuracy_score(yt_,(yp_>=0.5).astype(int))*len(grp)):>5} "
                  f"{accuracy_score(yt_,(yp_>=0.5).astype(int)):>7.4f} "
                  f"{brier_score_loss(yt_,yp_):>7.4f}")

        # Upset analysis
        val['Underdog1']     = val['S1'] > val['S2']
        val['UpsetOccurred'] = (( val['Underdog1']&(val['Actual']==1)) |
                                (~val['Underdog1']&(val['Actual']==0)))
        val['PickedUpset']   = (( val['Underdog1']&(val['Pred']>=0.5)) |
                                (~val['Underdog1']&(val['Pred']< 0.5)))
        nu = val['UpsetOccurred'].sum()
        nc = (val['UpsetOccurred']&val['PickedUpset']).sum()
        nf = (~val['UpsetOccurred']&val['PickedUpset']).sum()
        print(f"\n  UPSET ANALYSIS")
        print(f"  Upsets : {nu}/{len(val)}  "
              f"Caught : {nc}  Recall {100*nc/max(nu,1):.1f}%  "
              f"False alarms : {nf}  Rate {100*nf/max(len(val)-nu,1):.1f}%")

        # Top 10 misses
        val['T1Name']  = val['T1'].apply(tname)
        val['T2Name']  = val['T2'].apply(tname)
        val['Brier_i'] = (val['Actual']-val['Pred'])**2
        val['Winner']  = val.apply(
            lambda r: r['T1Name'] if r['Actual']==1 else r['T2Name'], axis=1)
        val['Correct'] = yt == yb
        print(f"\n  TOP 10 BIGGEST MISSES")
        print(f"  {'Round':<14} {'T1':<18} {'T2':<18} "
              f"{'Pred':>6} {'Act':>4} {'Winner':<18} {'OK':>3} {'B':>6}")
        print("  " + "─" * 90)
        for _, r in val.nlargest(10,'Brier_i').iterrows():
            print(f"  {r['Round']:<14} {r['T1Name']:<18} {r['T2Name']:<18} "
                  f"{r['Pred']:>6.3f} {int(r['Actual']):>4} "
                  f"{r['Winner']:<18} "
                  f"{'✓' if r['Correct'] else '✗':>3} "
                  f"{r['Brier_i']:>6.4f}")

        # Final scorecard
        print("\n" + "═" * 62)
        print("  FINAL SCORECARD  (v9 — AutoGluon + HPO)")
        print("═" * 62)
        print(f"  Train         : {TRAIN_SEASONS[0]}–{TRAIN_SEASONS[-1]}  "
              f"(zero leakage)")
        print(f"  Test          : {PRED_SEASON} tournament")
        print(f"  AutoGluon     : preset={PRESET}  "
              f"time_limit={TIME_LIMIT}s")
        print(f"  Best model    : {best_model}")
        print(f"  Val log_loss  : {best_score:.4f}")
        print(f"  Stacking      : 2 levels, 8-fold bagging, "
              f"weighted ensemble")
        print(f"  HPO           : Bayesian search, "
              f"{HPO_TRIALS} trials/model")
        print(f"  Features      : 29  (winning Kaggle feature set)")
        print(f"  Brier Score   : {brier:.4f}  "
              f"(baseline {bl_b:.4f})")
        print(f"  Skill Score   : {skill:.4f}")
        print(f"  Log Loss      : {ll:.4f}")
        print(f"  ROC-AUC       : {auc:.4f}")
        print(f"  Accuracy      : {acc:.4f}  "
              f"({int(acc*len(val))}/{len(val)} correct)")
        print(f"  Upsets Caught : {nc}/{nu}  "
              f"({100*nc/max(nu,1):.1f}% recall)")
        print("═" * 62)

        out = ['ID','Round','T1Name','T2Name','S1','S2',
               'Pred','Actual','Winner','Correct','Brier_i']
        vp  = f'{BASE}validation_{PRED_SEASON}_v9_detailed.csv'
        val[out].sort_values('Brier_i',ascending=False).to_csv(vp,index=False)
        print(f"\n  Detailed CSV → {vp}")
        print(f"  AutoGluon models saved → {AG_SAVE_PATH}")

print("\n  Done.")

AutoGluon already installed ✓
  NCAA March Madness v9 — AutoGluon + HPO

  Preset     : best_quality
  Time limit : 3600s (60 min)
  HPO trials : 15 per model type

[1/7] Loading data...
     Regular season rows : 122,775
     Tournament rows     : 1,449
     Train seasons       : 2013–2023
     Predict / Test      : 2024

[2/7] Computing ELO ratings (regular-season only, pre-tournament)...
     7,702 (season, team) ELO values
[3/7] Computing GLM team quality scores (raddar method)...
     7,617 (season, team) quality scores
[4/7] Building per-team season averages (regular-season only)...
     7,617 (season, team) stat profiles
[5/7] Building matchup rows (29-feature set)...
     Building tournament matchup rows...
     Building regular-season sample rows...
     Tournament matchups (2013-2023)  : 669
     Reg-season sample   (2013-2023)  : 17,339
     AutoGluon train set              : 16,207
     AutoGluon validation set         : 1,801
     Features                         : 29

  P

ModuleNotFoundError: No module named 'autogluon.core.space'